In [1]:
import pandas as pd

In [2]:
path_og = "./internal_train_crop_0.4.csv"
path_fps = "./fps_fixed_done.csv"

df = pd.read_csv(path_og)

In [3]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_final = df[df_rsna.columns]

df_final = pd.concat([df_final, df_rsna], ignore_index=True)
#df_final.to_csv("./internal_train_crop_0.4_rsna.csv", index=False)

In [4]:
df

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis,iom_artery,iom_vein
0,Tr0001.nii.gz,253.0,231.0,141.0,19,21,13,aneurysm,1499.0,12.411585,21.317637,0.907939,0.010674
1,Tr0002.nii.gz,236.0,196.5,213.5,15,12,14,aneurysm,975.0,10.408462,17.395071,0.998974,0.000000
2,Tr0003.nii.gz,315.5,326.5,75.5,20,16,16,aneurysm,1659.0,12.907482,19.155333,0.990958,0.000000
3,Tr0004.nii.gz,298.0,237.0,166.5,7,7,6,aneurysm,168.0,6.285128,8.259330,1.000000,0.000000
4,Tr0005.nii.gz,249.0,214.5,165.5,9,8,12,aneurysm,401.0,7.154678,13.706594,0.962594,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1368,Tr1183.nii.gz,281.0,276.5,193.0,13,16,13,aneurysm,1253.0,10.679110,18.547994,0.723065,0.002394
1369,Tr1184.nii.gz,269.5,301.0,189.5,8,7,4,aneurysm,165.0,4.458395,9.245159,0.236364,0.763636
1370,Tr1185.nii.gz,280.0,264.5,214.0,21,22,23,aneurysm,3251.0,12.740924,30.090392,0.866195,0.082436
1371,Tr1186.nii.gz,328.0,261.5,196.0,15,12,11,aneurysm,858.0,9.711334,17.275426,0.933566,0.041958


In [5]:
df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]
df_fps_og

/tmp/ipykernel_2688547/4286331924.py:1: DtypeWarning: Columns (11,12,13,14,15,16,19,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]


,seriesuid,probability,coordX,coordY,coordZ,d,h,w,is_fp,Comments,Aneurysm,Jisoo review (y-if agreed w anatomy),comments,GY Comments,is_FP,is_aneurysm,is_infundibulum,needs_review
0,Tr0001.nii.gz,0.992386,264.82675,304.70288,55.607536,15.581342,12.330435,12.551337,1.0,calcification of RV4,RICA-C7,y,NaN,NaN,1,NaN,NaN,NaN
1,Tr0001.nii.gz,0.945425,207.39828,205.01656,160.953900,5.864290,8.284704,8.960328,1.0,branch of RMCA-M1,NaN,y,NaN,NaN,1,NaN,NaN,NaN
2,Tr0002.nii.gz,0.972559,236.02684,288.35358,130.364720,12.565471,14.651035,14.723995,1.0,branch of right Posterior Inferior Cerebellar ...,RICA-C7,vein in post fossa,NaN,NaN,1,NaN,NaN,NaN
3,Tr0002.nii.gz,0.963586,347.29272,223.22868,8.606861,37.163760,30.523613,29.429272,1.0,not found (Left ICA-C2),NaN,y,NaN,NaN,1,NaN,NaN,NaN
4,Tr0002.nii.gz,0.948249,367.75235,225.88980,51.977787,9.012156,11.282431,11.766287,1.0,vessel of left face,NaN,y,NaN,NaN,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,Tr1185.nii.gz,0.852029,201.01909,250.76064,258.549350,7.378265,9.299912,9.971880,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1406,Tr1185.nii.gz,0.830833,188.24872,259.54950,64.345290,6.537921,8.895191,9.611685,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1407,Tr1186.nii.gz,0.965628,248.64972,283.99197,67.101830,10.471547,15.423156,15.434340,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1408,Tr1186.nii.gz,0.964017,358.46545,282.26907,27.951757,7.773346,9.695757,10.266037,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Remove only annotated aneurysms 

In [ ]:
def remove_aneurysms_but_keep_non_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return False
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return True
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return True
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return True


df_fps = df_fps_og.copy()
#remove all entries for which is_aneurysm, is_infundibulum and is_FP are all None

df_fps["keep"] = df_fps.apply(remove_aneurysms_but_keep_non_reviewed, axis=1)
df_fps_non_r = df_fps[df_fps["keep"]]
df_fps_non_r["lesion"] = "non_aneurysm"
df_fps_non_r

/tmp/ipykernel_951033/2709879760.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r["lesion"] = "non_aneurysm"


,seriesuid,probability,coordX,coordY,coordZ,d,h,w,is_fp,Comments,Aneurysm,Jisoo review (y-if agreed w anatomy),comments,GY Comments,is_FP,is_aneurysm,is_infundibulum,needs_review,keep,lesion
0,Tr0001.nii.gz,0.992386,264.82675,304.70288,55.607536,15.581342,12.330435,12.551337,1.0,calcification of RV4,RICA-C7,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
1,Tr0001.nii.gz,0.945425,207.39828,205.01656,160.953900,5.864290,8.284704,8.960328,1.0,branch of RMCA-M1,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
2,Tr0002.nii.gz,0.972559,236.02684,288.35358,130.364720,12.565471,14.651035,14.723995,1.0,branch of right Posterior Inferior Cerebellar ...,RICA-C7,vein in post fossa,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
3,Tr0002.nii.gz,0.963586,347.29272,223.22868,8.606861,37.163760,30.523613,29.429272,1.0,not found (Left ICA-C2),NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
4,Tr0002.nii.gz,0.948249,367.75235,225.88980,51.977787,9.012156,11.282431,11.766287,1.0,vessel of left face,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,Tr1185.nii.gz,0.852029,201.01909,250.76064,258.549350,7.378265,9.299912,9.971880,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,non_aneurysm
1406,Tr1185.nii.gz,0.830833,188.24872,259.54950,64.345290,6.537921,8.895191,9.611685,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,non_aneurysm
1407,Tr1186.nii.gz,0.965628,248.64972,283.99197,67.101830,10.471547,15.423156,15.434340,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,non_aneurysm
1408,Tr1186.nii.gz,0.964017,358.46545,282.26907,27.951757,7.773346,9.695757,10.266037,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,non_aneurysm


In [35]:
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None
df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r

/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis,iom_artery,iom_vein
0,Tr0001.nii.gz,264.82675,304.70288,55.607536,12.551337,12.330435,15.581342,non_aneurysm,None,None,None,None,None
1,Tr0001.nii.gz,207.39828,205.01656,160.953900,8.960328,8.284704,5.864290,non_aneurysm,None,None,None,None,None
2,Tr0002.nii.gz,236.02684,288.35358,130.364720,14.723995,14.651035,12.565471,non_aneurysm,None,None,None,None,None
3,Tr0002.nii.gz,347.29272,223.22868,8.606861,29.429272,30.523613,37.163760,non_aneurysm,None,None,None,None,None
4,Tr0002.nii.gz,367.75235,225.88980,51.977787,11.766287,11.282431,9.012156,non_aneurysm,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,Tr1185.nii.gz,201.01909,250.76064,258.549350,9.971880,9.299912,7.378265,non_aneurysm,None,None,None,None,None
1406,Tr1185.nii.gz,188.24872,259.54950,64.345290,9.611685,8.895191,6.537921,non_aneurysm,None,None,None,None,None
1407,Tr1186.nii.gz,248.64972,283.99197,67.101830,15.434340,15.423156,10.471547,non_aneurysm,None,None,None,None,None
1408,Tr1186.nii.gz,358.46545,282.26907,27.951757,10.266037,9.695757,7.773346,non_aneurysm,None,None,None,None,None


In [36]:
# combine with original df

df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv("./internal_train_crop_0.4_mixed_fps_done.csv", index=False)

/tmp/ipykernel_951033/4210680213.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [37]:
df_combined_non_r

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis,iom_artery,iom_vein
0,Tr0001.nii.gz,253.00000,231.00000,141.000000,19.000000,21.000000,13.000000,aneurysm,1499.0,12.411585,21.317637,0.907939,0.010674
1,Tr0002.nii.gz,236.00000,196.50000,213.500000,15.000000,12.000000,14.000000,aneurysm,975.0,10.408462,17.395071,0.998974,0.000000
2,Tr0003.nii.gz,315.50000,326.50000,75.500000,20.000000,16.000000,16.000000,aneurysm,1659.0,12.907482,19.155333,0.990958,0.000000
3,Tr0004.nii.gz,298.00000,237.00000,166.500000,7.000000,7.000000,6.000000,aneurysm,168.0,6.285128,8.259330,1.000000,0.000000
4,Tr0005.nii.gz,249.00000,214.50000,165.500000,9.000000,8.000000,12.000000,aneurysm,401.0,7.154678,13.706594,0.962594,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2687,Tr1185.nii.gz,201.01909,250.76064,258.549350,9.971880,9.299912,7.378265,non_aneurysm,NaN,NaN,NaN,NaN,NaN
2688,Tr1185.nii.gz,188.24872,259.54950,64.345290,9.611685,8.895191,6.537921,non_aneurysm,NaN,NaN,NaN,NaN,NaN
2689,Tr1186.nii.gz,248.64972,283.99197,67.101830,15.434340,15.423156,10.471547,non_aneurysm,NaN,NaN,NaN,NaN,NaN
2690,Tr1186.nii.gz,358.46545,282.26907,27.951757,10.266037,9.695757,7.773346,non_aneurysm,NaN,NaN,NaN,NaN,NaN


In [18]:
df_rsna.columns

Index(['seriesuid', 'coordX', 'coordY', 'coordZ', 'd', 'h', 'w', 'lesion',
       'volume', 'major_axis', 'minor_axis'],
      dtype='object')

In [38]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv("./internal_train_crop_0.4_mixed_fps_done_rsna.csv", index=False)

In [39]:
df_rsna.head(1)

,seriesuid,coordX,coordY,coordZ,d,h,w,lesion,volume,maj_axis,min_axis
0,1.2.826.0.1.3680043.8.498.10005158603912009425...,283.0,275.0,162.0,6.740605,8.435049,9.386142,aneurysm,533.670969,9.386142,6.740605


In [40]:
df_combined_non_r.head(1).iloc[:, 0:20]

,seriesuid,coordX,coordY,coordZ,d,h,w,lesion,volume,maj_axis,min_axis
0,Tr0001.nii.gz,253.0,231.0,141.0,13.0,21.0,19.0,aneurysm,1499.0,21.317637,12.411585


In [41]:
def remove_aneurysms_and_not_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return False
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return True
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return True
    # if either of these are NaN, we need to check the Aneurysm column
    if (
        pd.isna(row["is_aneurysm"])
        and pd.isna(row["is_FP"])
        and pd.isna(row["is_infundibulum"])
    ):
        return False


df_fps = df_fps_og.copy()
df_fps["keep"] = df_fps.apply(remove_aneurysms_and_not_reviewed, axis=1)
df_fps_non_r = df_fps[df_fps["keep"]]
df_fps_non_r["lesion"] = "non_aneurysm"
df_fps_non_r

/tmp/ipykernel_951033/1121060062.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r["lesion"] = "non_aneurysm"


,seriesuid,probability,coordX,coordY,coordZ,d,h,w,is_fp,Comments,Aneurysm,Jisoo review (y-if agreed w anatomy),comments,GY Comments,is_FP,is_aneurysm,is_infundibulum,needs_review,keep,lesion
0,Tr0001.nii.gz,0.992386,264.82675,304.70288,55.607536,15.581342,12.330435,12.551337,1.0,calcification of RV4,RICA-C7,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
1,Tr0001.nii.gz,0.945425,207.39828,205.01656,160.953900,5.864290,8.284704,8.960328,1.0,branch of RMCA-M1,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
2,Tr0002.nii.gz,0.972559,236.02684,288.35358,130.364720,12.565471,14.651035,14.723995,1.0,branch of right Posterior Inferior Cerebellar ...,RICA-C7,vein in post fossa,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
3,Tr0002.nii.gz,0.963586,347.29272,223.22868,8.606861,37.163760,30.523613,29.429272,1.0,not found (Left ICA-C2),NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
4,Tr0002.nii.gz,0.948249,367.75235,225.88980,51.977787,9.012156,11.282431,11.766287,1.0,vessel of left face,NaN,y,NaN,NaN,1,NaN,NaN,NaN,True,non_aneurysm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,Tr0358.nii.gz,0.996608,253.70757,303.55060,86.311676,25.603682,21.306190,21.753223,1.0,NaN,NaN,NaN,NaN,Extra cranial carotid internal,1P,NaN,NaN,NaN,True,non_aneurysm
497,Tr0358.nii.gz,0.915927,354.58620,229.98796,160.848820,5.785738,7.962148,8.748288,1.0,NaN,NaN,NaN,NaN,"False positive of the left crinoid ICA, somewh...",1P,NaN,NaN,NaN,True,non_aneurysm
498,Tr0360.nii.gz,0.870993,347.98236,200.60887,138.641770,5.588859,8.082442,9.058777,1.0,NaN,NaN,NaN,NaN,"False positive at the left MCA m1 m2 junction,...",1P,NaN,NaN,NaN,True,non_aneurysm
499,Tr0360.nii.gz,0.853835,308.04750,291.43103,1.991226,12.270049,20.687748,20.290630,1.0,NaN,NaN,NaN,NaN,Extra cranial,1P,NaN,NaN,NaN,True,non_aneurysm


In [42]:
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None
df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r

/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fps_non_r[col] = None
/tmp/ipykernel_951033/4186681180.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis,iom_artery,iom_vein
0,Tr0001.nii.gz,264.82675,304.70288,55.607536,12.551337,12.330435,15.581342,non_aneurysm,None,None,None,None,None
1,Tr0001.nii.gz,207.39828,205.01656,160.953900,8.960328,8.284704,5.864290,non_aneurysm,None,None,None,None,None
2,Tr0002.nii.gz,236.02684,288.35358,130.364720,14.723995,14.651035,12.565471,non_aneurysm,None,None,None,None,None
3,Tr0002.nii.gz,347.29272,223.22868,8.606861,29.429272,30.523613,37.163760,non_aneurysm,None,None,None,None,None
4,Tr0002.nii.gz,367.75235,225.88980,51.977787,11.766287,11.282431,9.012156,non_aneurysm,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,Tr0358.nii.gz,253.70757,303.55060,86.311676,21.753223,21.306190,25.603682,non_aneurysm,None,None,None,None,None
497,Tr0358.nii.gz,354.58620,229.98796,160.848820,8.748288,7.962148,5.785738,non_aneurysm,None,None,None,None,None
498,Tr0360.nii.gz,347.98236,200.60887,138.641770,9.058777,8.082442,5.588859,non_aneurysm,None,None,None,None,None
499,Tr0360.nii.gz,308.04750,291.43103,1.991226,20.290630,20.687748,12.270049,non_aneurysm,None,None,None,None,None


In [43]:
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done.csv", index=False
)

/tmp/ipykernel_951033/2098193903.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [44]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv("./internal_train_crop_0.4_confirmed_fps_done_rsna.csv", index=False)

In [45]:
path_og = "./internal_train_crop_0.4.csv"
path_fps = "./fps_fixed_done.csv"

df = pd.read_csv(path_og)

df_fps = df_fps_og.copy()

In [46]:
def keep_aneurysms_and_keep_non_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return "aneurysm"
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return "non_aneurysm"
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return "non_aneurysm"
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return "non_aneurysm"


df_fps = df_fps_og.copy()
df_fps["lesion"] = df_fps.apply(keep_aneurysms_and_keep_non_reviewed, axis=1)
df_fps_non_r = df_fps.copy()
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None

df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r["lesion"].value_counts()

lesion
non_aneurysm    1319
aneurysm          91
Name: count, dtype: int64

In [47]:
df_fps["is_aneurysm"].value_counts()

is_aneurysm
1.0    91
Name: count, dtype: int64

In [48]:
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_fps_done_extra_aneu.csv", index=False
)

/tmp/ipykernel_951033/1178979446.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [50]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv(
    "./internal_train_crop_0.4_mixed_fps_done_extra_aneu_rsna.csv", index=False
)

In [51]:
import pandas as pd
import numpy as np

In [52]:
path_og = "./internal_train_crop_0.4.csv"
path_fps = "./fps_fixed_done.csv"

df = pd.read_csv(path_og)
df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]
df_fps = df_fps_og.copy()

/tmp/ipykernel_951033/2281205112.py:5: DtypeWarning: Columns (11,12,13,14,15,16,19,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fps_og = pd.read_csv(path_fps).iloc[:, 2:20]


In [53]:
def keep_aneurysms_and_keep_only_reviewed(row):
    # if jisoo marked as aneurysm then remove
    if row["is_aneurysm"] == 1:
        return "aneurysm"
    # if jisoo marked as non-aneurysm then keep
    elif row["is_FP"] == "1" or row["is_FP"] == "1P":
        return "non_aneurysm"
    # if jisoo marked as infundibulum then discard
    elif row["is_infundibulum"] == 1:
        return "non_aneurysm"
    # if either of these are NaN, we need to check the Aneurysm column
    else:
        return "remove"


df_fps = df_fps_og.copy()
df_fps["lesion"] = df_fps.apply(keep_aneurysms_and_keep_only_reviewed, axis=1)
df_fps_non_r = df_fps.copy()
cols_to_add = ["volume", "min_axis", "maj_axis", "iom_artery", "iom_vein"]
for col in cols_to_add:
    if col not in df_fps_non_r.columns:
        df_fps_non_r[col] = None

df_fps_non_r = df_fps_non_r[df.columns]
df_fps_non_r["lesion"].value_counts()

lesion
remove          923
non_aneurysm    396
aneurysm         91
Name: count, dtype: int64

In [54]:
df_fps_non_r = df_fps_non_r[df_fps_non_r["lesion"] != "remove"]
df_fps["is_aneurysm"].value_counts()
df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)
df_combined_non_r.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done_extra_aneu.csv", index=False
)
# -*- coding: utf-8 -*-

/tmp/ipykernel_951033/3527990590.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined_non_r = pd.concat([df, df_fps_non_r], ignore_index=True)


In [55]:
path_rsna = "source_files_rsna/annotations_final.csv"
df_rsna = pd.read_csv(path_rsna)
df_rsna.columns = [
    "seriesuid",
    "coordX",
    "coordY",
    "coordZ",
    "d",
    "h",
    "w",
    "lesion",
    "volume",
    "maj_axis",
    "min_axis",
]
df_combined_non_r = df_combined_non_r[df_rsna.columns]

df_final = pd.concat([df_combined_non_r, df_rsna], ignore_index=True)
df_final.to_csv(
    "./internal_train_crop_0.4_confirmed_fps_done_extra_aneu_rsna.csv", index=False
)